# Eksplorativna analiza podataka i evaluacija ASR sistema nad RTS korpusom

> Ovaj notebook predstavlja reproduktivni eksperimentalni protokol za analizu podataka i evaluaciju modela automatskog prepoznavanja govora na srpskom jeziku. Analiza je postavljena tako da se svi zaključci generišu izračunavanjem nad aktuelnim ulaznim podacima u Kaggle okruženju, bez oslanjanja na ručno upisane rezultate.

Istraživački fokus obuhvata tri nivoa: kvalitet ulaznih podataka (akvizicija i poravnanje), akustičko-lingvistička svojstva korpusa i efekat fine-tuning postupka na strogo uparenom holdout skupu. Time se obezbeđuje metodološka veza između obrade podataka i performansi modela, što je centralni zahtev predmeta i završnog izveštaja.

## 1) Reproduktivni eksperimentalni setup
U ovoj sekciji se standardizuju putanje do podataka i koda, proverava integritet ulaza i obezbeđuje jedinstveno okruženje za izvršavanje svih narednih analiza. Cilj je da rezultati budu ponovljivi i nezavisni od lokalne konfiguracije računara.

In [ ]:
# Setup: resolve Kaggle input paths directly (no copy/extract to /kaggle/working)
import sys
from pathlib import Path

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT = Path('/kaggle/input')

def _find_first_existing(paths: list[Path]) -> Path | None:
    for p in paths:
        if p.exists():
            return p
    return None

def _find_named_dirs(root: Path, name: str, limit: int = 200) -> list[Path]:
    out: list[Path] = []
    if not root.exists():
        return out
    for p in root.rglob(name):
        if p.is_dir() and p.name == name:
            out.append(p)
            if len(out) >= limit:
                break
    return out

def _run_v1_score(p: Path) -> int:
    score = 0
    for sub in ['metrics', 'aligned_train', 'aligned_holdout']:
        if (p / sub).exists():
            score += 1
    if (p / 'metrics' / 'baseline_holdout_metrics.json').exists():
        score += 2
    if (p / 'metrics' / 'finetuned_holdout_metrics.json').exists():
        score += 2
    return score

print('=== Resolving ASR run (asr_full_run_v1) from Kaggle input ===')
run_candidates = [
    KAGGLE_INPUT / 'notebooks' / 'milomilanovi' / 'siap-14-march' / 'asr_full_run_v1',
    KAGGLE_INPUT / 'datasets' / 'milomilanovi' / 'asr-full-run-v1' / 'asr_full_run_v1',
    KAGGLE_INPUT / 'datasets' / 'milomilanovi' / 'asr_full_run_v1' / 'asr_full_run_v1',
    KAGGLE_INPUT / 'asr_full_run_v1',
    KAGGLE_INPUT / 'asr-full-run-v1' / 'asr_full_run_v1',
    KAGGLE_WORKING / 'new_res' / 'asr_full_run_v1',
    KAGGLE_WORKING / 'asr_full_run_v1',
]
RUN_V1_RESOLVED = _find_first_existing(run_candidates)

if RUN_V1_RESOLVED is None:
    discovered = _find_named_dirs(KAGGLE_INPUT, 'asr_full_run_v1')
    if discovered:
        discovered = sorted(discovered, key=_run_v1_score, reverse=True)
        RUN_V1_RESOLVED = discovered[0]

if RUN_V1_RESOLVED is None:
    raise RuntimeError(
        'asr_full_run_v1 folder nije pronađen u Kaggle input-u. '
        'Pokreni dijagnostičku ćeliju i prosledi PATH REPORT.'
    )
print(f'✓ RUN_V1 resolved: {RUN_V1_RESOLVED}')

print('\n=== Resolving code bundle (kaggle_bundle) from Kaggle input ===')
bundle_candidates = [
    KAGGLE_INPUT / 'datasets' / 'milomilanovi' / 'kaggle-bundle' / 'kaggle_bundle',
    KAGGLE_INPUT / 'kaggle_bundle' / 'kaggle_bundle',
    KAGGLE_INPUT / 'kaggle-bundle' / 'kaggle_bundle',
    KAGGLE_WORKING / 'kaggle_bundle',
]
BUNDLE_ROOT_RESOLVED = _find_first_existing(bundle_candidates)
if BUNDLE_ROOT_RESOLVED is None:
    found_bundle = _find_named_dirs(KAGGLE_INPUT, 'kaggle_bundle')
    if found_bundle:
        BUNDLE_ROOT_RESOLVED = found_bundle[0]
if BUNDLE_ROOT_RESOLVED is None:
    raise RuntimeError('kaggle_bundle folder nije pronađen u input-u.')

EDA_SRC = BUNDLE_ROOT_RESOLVED / 'src'
if not EDA_SRC.exists():
    raise RuntimeError(f'Bundle src ne postoji: {EDA_SRC}')
if str(EDA_SRC) in sys.path:
    sys.path.remove(str(EDA_SRC))
sys.path.insert(0, str(EDA_SRC))
print(f'✓ Code bundle resolved: {BUNDLE_ROOT_RESOLVED}')
print(f'✓ Added to sys.path (priority): {EDA_SRC}')

print('\n=== Resolving raw dataset (speech-recognation-raw) from Kaggle input ===')
raw_candidates = [
    KAGGLE_INPUT / 'datasets' / 'milomilanovi' / 'speech-recognation-raw' / 'raw',
    KAGGLE_INPUT / 'datasets' / 'milomilanovi' / 'speech-recognation-raw' / 'speech-recognation-raw' / 'raw',
    KAGGLE_INPUT / 'speech-recognation-raw' / 'raw',
    KAGGLE_INPUT / 'speech-recognition-raw' / 'raw',
    KAGGLE_WORKING / 'data' / 'raw',
    KAGGLE_WORKING / 'raw',
]
RAW_DIR_RESOLVED = _find_first_existing(raw_candidates)
if RAW_DIR_RESOLVED is None:
    found_raw = _find_named_dirs(KAGGLE_INPUT, 'raw')
    for p in found_raw:
        if (p / 'politika').exists() or (p / 'sport').exists():
            RAW_DIR_RESOLVED = p
            break
if RAW_DIR_RESOLVED is None:
    print('⚠ raw folder nije pronađen; acquisition analiza može biti parcijalna')
else:
    print(f'✓ Raw dataset resolved: {RAW_DIR_RESOLVED}')

print('\n=== Verification ===')
print(f'run_v1 exists: {RUN_V1_RESOLVED.exists()}')
print(f'bundle root: {BUNDLE_ROOT_RESOLVED}')
print(f'raw dir: {RAW_DIR_RESOLVED if RAW_DIR_RESOLVED else "N/A"}')
print('run_v1 subfolders:', sorted([p.name for p in RUN_V1_RESOLVED.iterdir()])[:10])
if RAW_DIR_RESOLVED and RAW_DIR_RESOLVED.exists():
    print('raw categories:', sorted([p.name for p in RAW_DIR_RESOLVED.iterdir() if p.is_dir()]))
print('\n✓ Setup completed (direct input mode, no copy)')

In [ ]:
# Import EDA modula sa fallback-om za Kaggle putanje
from __future__ import annotations
import importlib
import sys
from pathlib import Path

if not any('kaggle-bundle/kaggle_bundle/src' in p or p.endswith('/kaggle_bundle/src') for p in sys.path):
    candidate_src = [
        Path('/kaggle/input/datasets/milomilanovi/kaggle-bundle/kaggle_bundle/src'),
        Path('/kaggle/input/kaggle_bundle/kaggle_bundle/src'),
        Path('/kaggle/working/kaggle_bundle/src'),
    ]
    for c in candidate_src:
        if c.exists():
            sys.path.insert(0, str(c))
            print(f'✓ Added fallback src path: {c}')
            break

for _module_name in [
    'eda_modules.acoustics',
    'eda_modules.acquisition',
    'eda_modules.context',
    'eda_modules.evaluation',
    'eda_modules.io_utils',
    'eda_modules.linguistics',
    'eda_modules.quality',
    'eda_modules.reporting',
    'eda_modules.stats',
    'eda_modules.taxonomy',
    'eda_modules.visualization',
]:
    if _module_name in sys.modules:
        importlib.reload(sys.modules[_module_name])

from eda_modules.acoustics import run_audio_analysis
from eda_modules.acquisition import plot_data_acquisition, run_data_acquisition
from eda_modules.context import build_context
from eda_modules.evaluation import (
    run_ablation_analysis,
    run_character_levenshtein,
    run_error_breakdown,
    run_fair_comparison,
    run_oov_analysis,
    run_statistical_tests,
)
from eda_modules.linguistics import run_text_analysis
from eda_modules.quality import run_quality_analysis
from eda_modules.reporting import build_results_digest
from eda_modules.taxonomy import run_taxonomy_analysis

print('✓ Svi EDA moduli uspesno importovani')

## 2) Analitički okvir i pravila poređenja
Svi kvantitativni zaključci u ovom radu zasnivaju se na velikom Kaggle run-u iz asr_full_run_v1 i na strogom sample_id uparivanju baseline i finetuned predikcija. Ovakav protokol eliminiše pristrasnost poređenja izazvanu promenom evaluacionog skupa i omogućava da se razlike u WER/CER interpretiraju kao posledica metodologije, a ne posledica različitih uzoraka.

U nastavku se analiza vodi od podataka ka modelu: najpre se ispituje struktura i kvalitet korpusa, zatim akustičke i lingvističke karakteristike, pa tek onda performanse modela i analiza grešaka. Time je redosled usklađen sa principom data-centric istraživanja.

## 3) Akvizicija i pokrivenost korpusa

In [ ]:
# Build context and override paths to use resolved Kaggle input locations
if RUN_V1_RESOLVED is None:
    raise RuntimeError('RUN_V1_RESOLVED nije postavljen. Pokreni prvo setup ćeliju.')

ctx = build_context(root=Path('/kaggle/working'))

# Force direct-input resolved paths (avoid assumptions about /kaggle/working copies)
ctx['RUN_V1_DIR'] = RUN_V1_RESOLVED
ctx['RUN_V2_DIR'] = RUN_V1_RESOLVED
ctx['NEW_RES_DIR'] = RUN_V1_RESOLVED.parent
ctx['DATA_RAW_DIR'] = RAW_DIR_RESOLVED if RAW_DIR_RESOLVED is not None else ctx.get('DATA_RAW_DIR')

# Metrics/predictions explicitly bound to resolved run
metrics_dir = RUN_V1_RESOLVED / 'metrics'
ctx['V1_BASELINE_METRICS'] = metrics_dir / 'baseline_holdout_metrics.json'
ctx['V1_FINETUNED_METRICS'] = metrics_dir / 'finetuned_holdout_metrics.json'
ctx['V1_BASELINE_PRED'] = metrics_dir / 'baseline_holdout_predictions.jsonl'
ctx['V1_FINETUNED_PRED'] = metrics_dir / 'finetuned_holdout_predictions.jsonl'
ctx['V2_BASELINE_METRICS'] = ctx['V1_BASELINE_METRICS']
ctx['V2_FINETUNED_METRICS'] = ctx['V1_FINETUNED_METRICS']
ctx['V2_BASELINE_PRED'] = ctx['V1_BASELINE_PRED']
ctx['V2_FINETUNED_PRED'] = ctx['V1_FINETUNED_PRED']

# Prioritize aligned folders from resolved run
aligned_candidates = [
    RUN_V1_RESOLVED / 'aligned_train',
    RUN_V1_RESOLVED / 'aligned_holdout',
    RUN_V1_RESOLVED / 'aligned_raw_v1',
    RUN_V1_RESOLVED / 'aligned_raw_v1_improved',
    RUN_V1_RESOLVED / 'aligned_raw_v1_rerun',
]
ctx['ALIGN_DIRS'] = [p for p in aligned_candidates if p.exists()]

print('✓ Context built (direct input mode):')
print(f"  ROOT: {ctx['ROOT']}")
print(f"  RUN_V1_DIR: {ctx['RUN_V1_DIR']}")
print(f"  DATA_RAW_DIR: {ctx['DATA_RAW_DIR']}")
print(f"  ALIGN_DIRS: {[p.as_posix() for p in ctx['ALIGN_DIRS']]}")

In [ ]:
# Analiza akvizicije i pokrivenosti + fail-fast validacija
import pandas as pd

acquisition = run_data_acquisition(ctx)
raw_df = acquisition['raw_df']
alignment_df = acquisition['alignment_df']
raw_total = acquisition['raw_total']
best_aligned = acquisition['best_aligned']

print('✓ Data acquisition analysis completed')

if alignment_df.empty:
    raise RuntimeError(
        'Akvizicija nije ucitala alignment evidenciju (alignment_df je prazan). '\
        'Proveri da li dataset sadrzi manifest.json/report.json ili audio/text foldere.'
    )

aligned_counts = pd.to_numeric(alignment_df.get('chunk_count', pd.Series(dtype=float)), errors='coerce')
valid_aligned = aligned_counts.fillna(0).max() > 0
if not valid_aligned:
    display(alignment_df)
    raise RuntimeError(
        'Akvizicija je vratila samo 0/NaN chunk_count vrednosti. '\
        'Notebook zaustavljen da bi se izbegli pogresni zakljucci o pokrivenosti korpusa.'
    )

if 'status' in alignment_df.columns and (alignment_df['status'] == 'fallback_from_files').any():
    print('⚠ Manifest nije pronadjen za deo skupova; chunk_count je procenjen iz audio/text fajlova.')

if {'suspicious_count', 'dropped_suspicious_count'}.issubset(alignment_df.columns):
    print('Napomena: suspicious_count = oznaceni sumnjivi chunkovi; dropped_suspicious_count = QA odbaceni chunkovi.')

display_cols = [
    'dataset',
    'chunk_count',
    'suspicious_count',
    'dropped_suspicious_count',
    'status',
]
existing_cols = [c for c in display_cols if c in alignment_df.columns]
display(alignment_df[existing_cols] if existing_cols else alignment_df)

In [ ]:
plot_data_acquisition(raw_df=raw_df, raw_total=raw_total, best_aligned=best_aligned)

### Tumačenje nalaza
Analiza akvizicije kvantifikuje odnos između prikupljenih sirovih zapisa i segmenata koji su zadržani nakon poravnanja i QA filtriranja. Ovaj odnos je ključan indikator kvaliteta supervision signala: ako je stopa odbacivanja visoka ili neravnomerno raspoređena po izvorima i periodima, model će učiti na distribuciji koja više ne odražava realni domen.

Zbog toga se rezultat ove sekcije ne posmatra samo deskriptivno, već kao dijagnostički instrument za planiranje narednog ciklusa prikupljanja i čišćenja podataka.

## 4) Akustička svojstva i robustnost signala

In [ ]:
# Akustička analiza
audio_section = run_audio_analysis(ctx)
audio_df = audio_section['audio_df']
zcr_error_bridge_stats = audio_section['zcr_error_bridge_stats']

print('✓ Audio analysis completed')
print(f'Analizirano {len(audio_df)} audio uzoraka')

### Tumačenje nalaza
Akustička analiza pokazuje da korpus nije homogen i da pojedini režimi signala sistematski utiču na tipove grešaka koje model pravi. Posebno je značajan nalaz da kombinacija visokog ZCR i nižeg SNR prati veću zastupljenost frikativnih supstitucija, što ukazuje na selektivnu fonetsku degradaciju, a ne samo na opšti rast greške.

Metodološka implikacija je da kontrola akustičkog kvaliteta mora biti deo data pipeline-a, jer direktno utiče na granice performansi modela u realnim terenskim uslovima.

## 5) Lingvistička analiza korpusa

In [ ]:
# Lingvistička analiza
text_section = run_text_analysis(ctx)
text_df = text_section['text_df']
all_tokens = text_section['all_tokens']

print('✓ Text analysis completed')
print('Analizirani tekstualni uzorci')

### Tumačenje nalaza
Lingvistički profil korpusa (raspodele dužina, script-mix, numerički i specijalni obrasci) predstavlja važan izvor varijanse evaluacionih metrika. Posmatrano iz ugla eksperimentalne validnosti, tekstualna normalizacija nije kozmetička obrada, već kontrolni mehanizam koji smanjuje šum merenja i omogućava stabilnije poređenje modela.

Zato se ova sekcija koristi kao veza između opisa podataka i kasnijih zaključaka o performansama.

## 6) Kvalitet podataka i detekcija outlier-a

In [ ]:
# Quality analiza
quality_section = run_quality_analysis(ctx)
quality_df = quality_section['quality_df']

print('✓ Quality analysis completed')
print(f'Analizirano {len(quality_df)} kvalitativnih uzoraka')

### Tumačenje nalaza
Distribucije CPS i aligned_word_ratio metrika potvrđuju da u podacima postoje strukturne anomalije koje bi, bez filtriranja, uvodile sistematsku grešku u obuku i evaluaciju. U tom smislu, QA pragovi nisu arbitrarni, već operacionalizacija kriterijuma validnog supervision signala.

Drugim rečima, ova sekcija formalizuje granicu između informativnih i štetnih uzoraka i direktno doprinosi stabilnosti narednih modelskih zaključaka.

## 7) Evaluacija sistema: Baseline naspram Finetuned modela

In [ ]:
# Ablation analiza - samo processed modus
ablation_section = run_ablation_analysis(ctx)
data_first_metrics = ablation_section['data_first_metrics']
normalization_effect = ablation_section['normalization_effect']
data_quality_evolution = ablation_section['data_quality_evolution']
quality_cost_summary = ablation_section['quality_cost_summary']
ablation_df = ablation_section['ablation_df']
improvement_attribution = ablation_section['improvement_attribution']

print('✓ Ablation analysis completed')

In [ ]:
# Fair comparison - striktno sample_id poređenje
comparison_section = run_fair_comparison(ctx=ctx, quality_df=quality_df)
pred_map = comparison_section['pred_map']
eval_alignment_df = comparison_section['eval_alignment_df']
v1_paired_df = comparison_section['v1_paired_df']

print('✓ Fair comparison completed')
print(f'Upareni uzorci: {len(v1_paired_df)}')

In [ ]:
# Statistički testovi
run_statistical_tests(eval_alignment_df=eval_alignment_df, v1_paired_df=v1_paired_df)

In [ ]:
# OOV analiza
run_oov_analysis(pred_map)

In [ ]:
# Error breakdown
run_error_breakdown(pred_map)

In [ ]:
# Character Levenshtein analiza
run_character_levenshtein(pred_map)

### Tumačenje nalaza
U ovoj fazi prethodni nalazi o kvalitetu podataka i signalu testiraju se kroz strogo upareno poređenje modela. Rezultati na istom skupu uzoraka pokazuju da finetuning donosi merljivo i statistički podržano poboljšanje, dok je istovremeno zadržana metodološka konzistentnost poređenja.

Time je obezbeđen ključni uslov akademski validne evaluacije: zaključak o superiornosti modela ne počiva na promeni test populacije, već na promeni modela.

## 8) Taksonomija grešaka i kvalitativna inspekcija

In [ ]:
# Taxonomy analiza
taxonomy_section = run_taxonomy_analysis(ctx=ctx, pred_map=pred_map, quality_df=quality_df)
taxonomy_df = taxonomy_section['taxonomy_df']

print('✓ Taxonomy analysis completed')

### Tumačenje nalaza
Taksonomska razgradnja pokazuje da greške nisu monolitne i da imaju različite uzroke: deo potiče iz strukture podataka, deo iz akustičkih uslova, a deo iz lingvističke složenosti domena. Ovakva dekompozicija je važna jer omogućava ciljane intervencije u narednim iteracijama umesto generičkog povećanja obima treninga.

Zato se ova sekcija koristi kao osnova za plan unapređenja sistema i za argumentaciju u završnom izveštaju.

## 9) Sinteza rezultata i ključni metrički pokazatelji

In [ ]:
# Build final digest i eksplicitno prikazi glavne rezultate
digest_section = build_results_digest(
    data_first_metrics=data_first_metrics,
    ablation_df=ablation_df,
    data_quality_evolution=data_quality_evolution,
    quality_cost_summary=quality_cost_summary,
    normalization_effect=normalization_effect,
    zcr_error_bridge_stats=zcr_error_bridge_stats,
)
digest = digest_section['digest']
summary_table_df = digest_section['summary_table_df']
latex_table = digest_section['latex_table']

print('✓ Results digest generated')
display(summary_table_df)
print('\nKljučni digest:')
for key in sorted(digest.keys()):
    print(f'- {key}: {digest[key]}')

### Inženjerski sažetak
Analiza je sprovedena na velikom, strogo uparenom holdout skupu, što omogućava metodološki korektno poređenje baseline i finetuned modela. Dobijeni rezultati potvrđuju statistički značajno poboljšanje finetuned varijante i pokazuju da je dominantan faktor stabilnosti sistema kvalitet ulaznih podataka i QA filtriranje.

Istovremeno, akustički i taksonomski nalazi ukazuju da preostale greške nisu slučajne, već povezane sa specifičnim režimima signala i fonetsko-lingvističkim osobinama domena. To određuje jasan pravac daljeg rada: ciljano proširenje i balansiranje podataka, uz održavanje strogog evaluacionog protokola po sample_id principu.

## 10) Automatski narativ i kontrola pokrivenosti zahteva
Naredne dve ćelije automatski generišu tekstualni sažetak i operativnu proveru pokrivenosti ključnih stavki iz zahteva predmeta. Na taj način se minimizuje rizik od ručno unetih, hardkodovanih zaključaka i obezbeđuje transparentna veza između rezultata i interpretacije.

In [ ]:
# Auto-narativ: tekst zasnovan na izračunatim metrikama, bez ručnog hardkodovanja
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

def _safe_float(v):
    try:
        return float(v)
    except Exception:
        return np.nan

def _fmt_num(value: float, precision: int = 3, nan_label: str = 'N/A') -> str:
    if value is None or not np.isfinite(value):
        return nan_label
    return f'{value:.{precision}f}'

# 1) Osnovne veličine skupa
n_quality = int(len(quality_df)) if 'quality_df' in globals() else 0
n_paired = int(len(v1_paired_df)) if 'v1_paired_df' in globals() else 0

# 2) WER sažetak iz digest-a
baseline_wer = _safe_float(digest.get('processed_baseline_wer', np.nan))
finetuned_wer = _safe_float(digest.get('processed_finetuned_wer', np.nan))
rel_impr = _safe_float(digest.get('processed_rel_improvement_percent', np.nan))

# 3) QA status
toxic_count = int(quality_df['is_toxic_sample'].sum()) if n_quality > 0 and 'is_toxic_sample' in quality_df.columns else 0
toxic_share = (100.0 * toxic_count / max(1, n_quality)) if n_quality > 0 else np.nan

# 4) Akustički bridge rezultat
if 'zcr_error_bridge_stats' in globals():
    zcr_share = _safe_float(zcr_error_bridge_stats.get('high_zcr_low_snr_mean_fricative_sub_share', np.nan))
    other_share = _safe_float(
        zcr_error_bridge_stats.get(
            'other_regimes_mean_fricative_sub_share',
            zcr_error_bridge_stats.get('others_mean_fricative_sub_share', np.nan),
        )
    )
    mw_p = _safe_float(zcr_error_bridge_stats.get('mann_whitney_p', np.nan))
else:
    zcr_share = np.nan
    other_share = np.nan
    mw_p = np.nan

# 5) Akvizicija validnost
if 'alignment_df' in globals() and isinstance(alignment_df, pd.DataFrame) and not alignment_df.empty:
    _aligned_counts = pd.to_numeric(alignment_df.get('chunk_count', pd.Series(dtype=float)), errors='coerce')
    acq_valid = bool(_aligned_counts.fillna(0).max() > 0)
else:
    acq_valid = False

narrative = f"""
### Automatski generisan narativ rezultata
U ovoj iteraciji analiza je izvršena nad kompletnim dostupnim podacima u Kaggle okruženju.
Quality audit obuhvatio je **{n_quality}** uzoraka, dok je strogo upareno poređenje baseline i finetuned sistema obavljeno nad **{n_paired}** uzoraka (sample_id join).

Na finalnom holdout poređenju dobijeno je: **WER baseline = {_fmt_num(baseline_wer, 4)}**, **WER finetuned = {_fmt_num(finetuned_wer, 4)}**.
Relativno poboljšanje finetuned sistema iznosi **{_fmt_num(rel_impr, 2)}%**.

U kvalitetu podataka identifikovano je **{toxic_count}** toksičnih uzoraka (**{_fmt_num(toxic_share, 2)}%**),
što potvrđuje da QA filtriranje ostaje ključna komponenta za stabilnu evaluaciju i interpretabilne metrike.

Akustički bridge pokazuje da je u režimu High-ZCR/Low-SNR prosečan udeo frikativnih supstitucija **{_fmt_num(zcr_share, 3)}**,
dok je kod ostalih režima **{_fmt_num(other_share, 3)}** (Mann-Whitney p={_fmt_num(mw_p, 4)}).
Ovaj nalaz empirijski povezuje akustičku degradaciju sa fonetskim tipom greške.

Validnost akvizicije/poravnanja u ovom run-u: **{'VALIDNO' if acq_valid else 'NEVALIDNO'}**.
"""
display(Markdown(narrative))

In [ ]:
# Checklist pokrivenosti zahteva predmeta (operativna forma)
from IPython.display import display
import numpy as np
import pandas as pd

if 'alignment_df' in globals() and isinstance(alignment_df, pd.DataFrame) and not alignment_df.empty:
    _aligned_counts = pd.to_numeric(alignment_df.get('chunk_count', pd.Series(dtype=float)), errors='coerce')
    acquisition_valid = bool(_aligned_counts.fillna(0).max() > 0)
else:
    acquisition_valid = False

zcr_bridge_valid = False
if 'zcr_error_bridge_stats' in globals() and isinstance(zcr_error_bridge_stats, dict):
    _zcr_main = zcr_error_bridge_stats.get('high_zcr_low_snr_mean_fricative_sub_share', np.nan)
    _zcr_other = zcr_error_bridge_stats.get(
        'other_regimes_mean_fricative_sub_share',
        zcr_error_bridge_stats.get('others_mean_fricative_sub_share', np.nan),
    )
    zcr_bridge_valid = bool(np.isfinite(_zcr_main) and np.isfinite(_zcr_other))

paired_eval_valid = bool('v1_paired_df' in globals() and len(v1_paired_df) > 0)
taxonomy_valid = bool('taxonomy_df' in globals() and len(taxonomy_df) > 0)
summary_valid = bool('summary_table_df' in globals() and len(summary_table_df) > 0)

checklist_rows = [
    ('Definicija i obim skupa podataka', bool(n_quality > 0)),
    ('Akvizicija i poravnanje imaju validne (nenulte) evidencije', acquisition_valid),
    ('Eksplorativna analiza podataka (akustika, lingvistika, quality)', bool('audio_df' in globals() and 'text_df' in globals() and 'quality_df' in globals())),
    ('Metod evaluacije i mera performansi (WER/CER)', summary_valid),
    ('Statističko testiranje razlika modela', paired_eval_valid),
    ('Analiza grešaka modela (OOV, breakdown, taxonomy)', taxonomy_valid),
    ('Akustički bridge ima numerički validne pokazatelje', zcr_bridge_valid),
    ('Reproducibilnost putanja i izvora podataka', bool('ctx' in globals() and 'RUN_V1_DIR' in ctx)),
]

checklist_df = pd.DataFrame(checklist_rows, columns=['Stavka zahteva', 'Pokriveno'])
checklist_df['Status'] = checklist_df['Pokriveno'].map({True: 'DA', False: 'NE'})
display(checklist_df[['Stavka zahteva', 'Status']])

missing = checklist_df.loc[~checklist_df['Pokriveno'], 'Stavka zahteva'].tolist()
if missing:
    print('\nNedostaje za punu pokrivenost:')
    for m in missing:
        print('-', m)
else:
    print('\n✓ Sve ključne stavke iz operativne checkliste su pokrivene u notebook-u.')